ICD-10 coding pipeline — Step 3: LLM code assignment from ranked candidates
*Co-authored with CoCo*

# Step 3: LLM Code Assignment (Matching)

## How it works

1. For each finding, we aggregate its top-N candidates (from Step 2) into a ranked list.
2. The LLM receives: the clinical finding + its structured metadata (category, acuity, laterality, causal_link, supporting quote) + the ranked candidate codes with descriptions.
3. The LLM selects the **single best code** from the candidates, or responds NONE if no candidate fits.
4. Output includes confidence score (0–1) and rationale for audit.

## Key design decisions
- **Grounded selection:** The LLM can ONLY choose from provided candidates (no hallucinated codes)
- **Context-rich prompt:** Acuity, laterality, causal_link, and supporting quote all help the LLM pick the most specific code
- **Confidence scoring:** Enables downstream thresholding (e.g., only report codes with confidence ≥ 0.75)
- **NONE option:** If no candidate fits, the finding is flagged rather than force-assigned a bad code

In [ ]:
%%sql -r ctx
-- Context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICD10_CODING_APP;
USE SCHEMA MATCHING;
USE WAREHOUSE COMPUTE_WH;
ALTER SESSION SET QUERY_TAG = 'icd10_v2:matching';

---
## Configuration

In [ ]:
%%sql -r config
-- ============================================================
-- MATCHING CONFIG
-- ============================================================
SET MATCHING_MODEL = 'claude-4-sonnet';
SET CONFIDENCE_THRESHOLD = 0.75;  -- Minimum confidence to count as a positive prediction

In [ ]:
%%sql -r prompt
-- Matching prompt
SET MATCHING_PROMPT = '
You are a certified medical coder. Select the BEST ICD-10-CM code for this clinical finding from the candidates.
Consider specificity, laterality, acuity, and clinical context.
If no candidate is appropriate, respond with NONE.
Respond ONLY with: {"code": "<ICD-10>", "description": "<desc>", "confidence": <0.0-1.0>, "rationale": "<brief>"}
';

---
## UDF: Assign ICD-10 Code

Takes a finding and its candidate list, returns the LLM's selection as JSON.

In [ ]:
%%sql -r create_udf
CREATE OR REPLACE FUNCTION MATCHING.ASSIGN_ICD10_CODE(
    FINDING VARCHAR,
    CATEGORY VARCHAR,
    ACUITY VARCHAR,
    LATERALITY VARCHAR,
    CAUSAL_LINK VARCHAR,
    SUPPORTING_QUOTE VARCHAR,
    CANDIDATES_JSON VARCHAR
)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    SNOWFLAKE.CORTEX.COMPLETE(
        $MATCHING_MODEL,
        CONCAT(
            $MATCHING_PROMPT, CHR(10), CHR(10),
            'Finding: ', FINDING, CHR(10),
            'Category: ', COALESCE(CATEGORY, 'unspecified'), CHR(10),
            'Acuity: ', COALESCE(ACUITY, 'unspecified'), CHR(10),
            'Laterality: ', COALESCE(LATERALITY, 'unspecified'), CHR(10),
            'Causal link: ', COALESCE(CAUSAL_LINK, 'none'), CHR(10),
            'Supporting quote: ', LEFT(COALESCE(SUPPORTING_QUOTE, ''), 500), CHR(10), CHR(10),
            'Candidates (ranked):', CHR(10),
            LEFT(CANDIDATES_JSON, 4000)
        )
    )
$$;

---
## Run Matching

In [ ]:
%%sql -r run_matching
-- Aggregate candidates per finding into JSON, then call UDF for assignment
CREATE OR REPLACE TABLE MATCHING.FINAL_ASSIGNMENTS_V2 AS
WITH candidates_agg AS (
    SELECT
        cm.FILE_NAME,
        cm.FINDING_SEQ,
        cm.FINDING,
        cm.CATEGORY,
        f.ACUITY,
        f.LATERALITY,
        cm.CAUSAL_LINK,
        f.SUPPORTING_QUOTE,
        ARRAY_AGG(
            OBJECT_CONSTRUCT('code', cm.ICD10_CODE, 'description', cm.DESCRIPTION, 'hcc', cm.HCC_CATEGORY, 'raf', cm.RAF_COEFFICIENT, 'rank', cm.FINAL_RANK)
        ) WITHIN GROUP (ORDER BY cm.FINAL_RANK) AS CANDIDATES
    FROM MATCHING.CANDIDATES_MERGED cm
    LEFT JOIN PROCESSING.ENCOUNTER_FINDINGS f
        ON cm.FILE_NAME = f.FILE_NAME AND cm.FINDING_SEQ = f.FINDING_SEQ
    GROUP BY cm.FILE_NAME, cm.FINDING_SEQ, cm.FINDING, cm.CATEGORY, f.ACUITY, f.LATERALITY, cm.CAUSAL_LINK, f.SUPPORTING_QUOTE
)
SELECT
    c.FILE_NAME,
    c.FINDING_SEQ,
    c.FINDING,
    c.CATEGORY,
    c.ACUITY,
    c.LATERALITY,
    c.CAUSAL_LINK,
    c.CANDIDATES,
    MATCHING.ASSIGN_ICD10_CODE(
        c.FINDING, c.CATEGORY, c.ACUITY, c.LATERALITY, c.CAUSAL_LINK,
        c.SUPPORTING_QUOTE, c.CANDIDATES::VARCHAR
    ) AS ASSIGNMENT_RAW,
    TRY_PARSE_JSON(REGEXP_SUBSTR(ASSIGNMENT_RAW, '\\{.*\\}', 1, 1, 's')):code::VARCHAR AS ASSIGNED_CODE,
    TRY_PARSE_JSON(REGEXP_SUBSTR(ASSIGNMENT_RAW, '\\{.*\\}', 1, 1, 's')):description::VARCHAR AS ASSIGNED_DESCRIPTION,
    TRY_PARSE_JSON(REGEXP_SUBSTR(ASSIGNMENT_RAW, '\\{.*\\}', 1, 1, 's')):confidence::FLOAT AS CONFIDENCE,
    TRY_PARSE_JSON(REGEXP_SUBSTR(ASSIGNMENT_RAW, '\\{.*\\}', 1, 1, 's')):rationale::VARCHAR AS RATIONALE,
    CURRENT_TIMESTAMP() AS ASSIGNED_AT
FROM candidates_agg c;

---
## Output Preview

In [ ]:
%%sql -r stats
-- Matching stats
SELECT
    COUNT(*) AS TOTAL_FINDINGS,
    COUNT_IF(ASSIGNED_CODE IS NOT NULL AND ASSIGNMENT_RAW NOT ILIKE '%NONE%') AS CODES_ASSIGNED,
    COUNT_IF(ASSIGNMENT_RAW ILIKE '%NONE%') AS NO_MATCH,
    ROUND(AVG(CONFIDENCE), 3) AS AVG_CONFIDENCE,
    COUNT_IF(CONFIDENCE >= $CONFIDENCE_THRESHOLD) AS ABOVE_THRESHOLD
FROM MATCHING.FINAL_ASSIGNMENTS_V2
WHERE ASSIGNED_CODE IS NOT NULL;

In [ ]:
%%sql -r sample
-- Sample assignments
SELECT FINDING, ASSIGNED_CODE, ASSIGNED_DESCRIPTION, CONFIDENCE, RATIONALE
FROM MATCHING.FINAL_ASSIGNMENTS_V2
WHERE ASSIGNED_CODE IS NOT NULL AND ASSIGNMENT_RAW NOT ILIKE '%NONE%'
ORDER BY CONFIDENCE DESC
LIMIT 15;

---
## Done

**Output:** `MATCHING.FINAL_ASSIGNMENTS_V2` — final ICD-10 code per finding with confidence

**Next:** Run `04_evaluation.ipynb` to evaluate against ground truth.